# 05 Models: LambdaMART, collaborative features, neural ranking

**Purpose.** Train the rankers of PROJECT_PLAN section 5 (item 4) one model family at a time and measure every step against the same references on the same searches. This part sets up the training pipeline, profiles memory and trains the **deployable non-personalised base** (E14). Ablations, collaborative features and neural models follow in later sections.

**Primary benchmark.** Lagged history popularity (`pop_history_lag1d`, history actions at least one day old): 0.337 NDCG@10 on `validation_clean` (E13). Differences below 0.003-0.005 are not treated as meaningful.

**Rules (PROJECT_PLAN 5-6, FINDINGS 2-3).**
- Inputs only from the groups in `features_v3_meta.json`; never `session_id`, `wid`, `label` or `n_candidates`.
- Training searches: the 40,000-search dev subsample without searches with a clamped final gap (post-search actions in the history, E12), without an engaged candidate (every order scores the same) and without single-candidate lists.
- Early stopping on an **inner set** of 10% of these training searches (whole searches, seed 4222), so validation is used only to report, never to pick the number of trees.
- LightGBM LambdaRank with **linear gains** `label_gain=[0,1,2,3]` (LightGBM's default gains are exponential, E06) and NDCG@10 for early stopping; final numbers come from our exact tie-aware evaluator (`scripts/evaluation.py`) on complete validation lists.
- Main population `validation_clean` (clamped searches removed); all validation searches as sensitivity. Model comparisons are paired per-search differences with bootstrap intervals over searches.
- Candidates were shuffled within each search when the features were built (notebook 03), so file order, which encodes the label, cannot reach a model.
- Missing values (unknown metadata) stay missing: LightGBM learns where to send them, instead of treating unknown as "no match".

**How to run.** Locally after notebooks 02 and 03 (they write `data/processed/`). On Colab: open this notebook from GitHub (branch `BRANCH` below); the first cell clones the public repository and mounts Drive. Copy `search_table.parquet`, `split_v1.csv`, `features_v3_meta.json` and the two `features_v3_*.parquet` files to `MyDrive/BT4222 Group Project/processed/` first. CPU is enough for this part.

In [1]:
from pathlib import Path
import gc
import json
import os
import sys
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = "feat/models"  # branch to clone on Colab; "main" once merged
REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None and IN_COLAB:  # opened from GitHub: clone the (public) repository
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/liamkunert/BT4222-group17-jdsearch.git", "/content/BT4222-group17-jdsearch"],
                   check=True)
    REPO_ROOT = Path("/content/BT4222-group17-jdsearch")
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    PROCESSED = Path("/content/drive/MyDrive/BT4222 Group Project/processed")
else:
    PROCESSED = REPO_ROOT / "data/processed"

sys.path.insert(0, str(REPO_ROOT))
from scripts.evaluation import RESULTS, evaluate, log_result, paired_difference, summarise

pd.options.display.float_format = "{:,.4f}".format
FEATURES = "v3"
SEED = 4222        # data decisions (inner split)
MODEL_SEED = 1     # LightGBM; three seeds only for finalists
meta = json.loads((PROCESSED / f"features_{FEATURES}_meta.json").read_text())
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
validation_ids = assignment.index[assignment.split == "validation"]
validation_sizes = searches.loc[validation_ids, "n_candidates"]
ALL_FEATURES = [f for group in meta["groups"].values() for f in group]
assert len(ALL_FEATURES) == len(set(ALL_FEATURES)) == 97
assert not set(ALL_FEATURES) & {"session_id", "wid", "label", "n_candidates"}
print(f"LightGBM {lgb.__version__}; {len(ALL_FEATURES)} features in {len(meta['groups'])} groups; processed data: {PROCESSED}")

LightGBM 4.6.0; 97 features in 13 groups; processed data: C:\Users\liamk\ClaudeProjects\BT4222\BT4222-group17-jdsearch\data\processed


## 1. Training population and inner early-stopping set

In [2]:
dev_ids = assignment.index[(assignment.split == "train") & assignment.dev_subsample]
dev_info = searches.loc[dev_ids]
drops = pd.DataFrame({
    "clamped final gap": dev_info.final_gap_clamped,
    "no engaged candidate": dev_info.n_engaged == 0,
    "single candidate": dev_info.n_candidates == 1,
})
train_pool = dev_ids[~drops.any(axis=1).to_numpy()]
rng = np.random.default_rng(SEED)
inner_ids = np.sort(rng.choice(train_pool.to_numpy(), size=round(0.1 * len(train_pool)), replace=False))
fit_ids = np.setdiff1d(train_pool.to_numpy(), inner_ids)
print(f"dev searches {len(dev_ids):,}; kept for training {len(train_pool):,} "
      f"(fit {len(fit_ids):,}, inner early-stopping set {len(inner_ids):,})")
drops.sum().rename("dropped searches (overlapping)").to_frame()

dev searches 40,000; kept for training 37,567 (fit 33,810, inner early-stopping set 3,757)


,dropped searches (overlapping)
clamped final gap,2116
no engaged candidate,503
single candidate,118


## 2. Loading and memory profile

Features are read **column by column into one pre-allocated float32 matrix** (column-major, which LightGBM accepts without copying), so no intermediate DataFrame of the full table is built. The rows of each search are contiguous in the feature files; LightGBM's `group` is the list of consecutive search sizes.

The profile loads all 97 features for the dev subsample and validation and builds a LightGBM dataset from them, which is what the full-model ablations need. "Peak" is the highest memory the process has used so far.

In [3]:
PROCESS = psutil.Process()


def memory_gb():
    """(current, peak) resident memory of this process in GB."""
    info = PROCESS.memory_info()
    peak = getattr(info, "peak_wset", None)  # Windows
    if peak is None:  # Linux / Colab: ru_maxrss is in KiB
        import resource
        peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024
    return info.rss / 1e9, peak / 1e9


def load_features(split, columns, sessions=None):
    """Session ids, float32 feature matrix (column-major) and labels; rows restricted to `sessions` if given."""
    path = PROCESSED / f"features_{FEATURES}_{split}.parquet"
    ids = pq.read_table(path, columns=["session_id"]).column(0).to_numpy()
    if not (np.diff(ids) >= 0).all():
        raise ValueError("Rows must be sorted by session_id so that every search is contiguous")
    keep = np.ones(len(ids), dtype=bool) if sessions is None else np.isin(ids, sessions)
    matrix = np.empty((int(keep.sum()), len(columns)), dtype=np.float32, order="F")
    for j, column in enumerate(columns):
        matrix[:, j] = pq.read_table(path, columns=[column]).column(0).to_numpy()[keep]
    labels = pq.read_table(path, columns=["label"]).column(0).to_numpy()[keep]
    return ids[keep], matrix, labels


def group_sizes(ids):
    """Sizes of consecutive runs of equal session ids (LightGBM's `group`)."""
    bounds = np.r_[0, np.flatnonzero(np.diff(ids)) + 1, len(ids)]
    return np.diff(bounds)

In [4]:
profile = [("start", *memory_gb())]
start = time.time()
train_sid, train_X, train_y = load_features("train_dev", ALL_FEATURES, sessions=train_pool)
profile.append(("train features loaded (97 columns)", *memory_gb()))
categorical = [ALL_FEATURES.index(c) for c in meta["categorical"]]
probe = lgb.Dataset(train_X, label=train_y, group=group_sizes(train_sid), categorical_feature=categorical,
                    free_raw_data=False, params={"verbose": -1, "feature_pre_filter": False}).construct()
profile.append(("LightGBM dataset built", *memory_gb()))
valid_sid, valid_X, valid_y = load_features("validation", ALL_FEATURES)
profile.append(("validation features loaded", *memory_gb()))
seconds = time.time() - start
train_rows, train_gb = train_X.shape[0], train_X.nbytes / 1e9
del probe, train_X, valid_X, train_sid, valid_sid, train_y, valid_y
gc.collect()
profile.append(("all freed", *memory_gb()))
profile = pd.DataFrame(profile, columns=["step", "current_gb", "peak_gb"]).set_index("step")
full_rows = train_rows * (assignment.split == "train").sum() / len(dev_ids)
print(f"training rows {train_rows:,} ({train_gb:.2f} GB as float32); load + build {seconds:.0f} s")
print(f"estimate for all training searches: {full_rows / 1e6:.1f}M rows, "
      f"{full_rows * len(ALL_FEATURES) * 4 / 1e9:.1f} GB as float32 before LightGBM's dataset")
print(f"machine memory: {psutil.virtual_memory().total / 1e9:.1f} GB total")
profile

training rows 3,391,571 (1.32 GB as float32); load + build 8 s
estimate for all training searches: 11.3M rows, 4.4 GB as float32 before LightGBM's dataset
machine memory: 16.9 GB total


,current_gb,peak_gb
step,,
start,0.2532,0.2613
train features loaded (97 columns),1.7782,1.7817
LightGBM dataset built,2.0654,2.0785
validation features loaded,2.7818,2.7836
all freed,0.4443,2.7836


## 3. E14: deployable non-personalised base (G0 + G1 + G2a)

Ten features: query length (G0, acts only through interactions because it is constant within a list), the eight query-title relevance features (G1) and lagged history popularity (G2a). Nothing in the base uses the shopper's history or same-day information, so it is the trained version of the deployable reference.

One fixed LightGBM configuration is used for all ablations (PROJECT_PLAN 5a); it is not tuned here. Success criterion, fixed before running: the lower bound of the paired 95% interval against lagged popularity on `validation_clean` is above +0.003.

In [5]:
PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "eval_at": [10],
    "label_gain": [0, 1, 2, 3],          # linear gains, as in our evaluator
    "lambdarank_truncation_level": 20,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_data_in_leaf": 500,
    "feature_pre_filter": False,
    "seed": MODEL_SEED,
    "deterministic": True,
    "force_row_wise": True,
    "num_threads": 4,
    "verbose": -1,
}
MAX_ROUNDS, PATIENCE = 3000, 100
BASE = meta["groups"]["G0_search"] + meta["groups"]["G1_relevance"] + meta["groups"]["G2a_popularity_deployable"]


def train_ranker(columns, params=PARAMS, label="", fit_sessions=None):
    """Fit on the fit searches (or a subset), stop early on the inner set; return the booster and a summary."""
    fit_sessions = fit_ids if fit_sessions is None else fit_sessions
    cats = [columns.index(c) for c in meta["categorical"] if c in columns]
    sid, X, y = load_features("train_dev", columns, sessions=fit_sessions)  # loaded separately: no copies of X
    fit_set = lgb.Dataset(X, label=y, group=group_sizes(sid), feature_name=columns, categorical_feature=cats)
    sid_inner, X_inner, y_inner = load_features("train_dev", columns, sessions=inner_ids)
    inner_set = lgb.Dataset(X_inner, label=y_inner, group=group_sizes(sid_inner), reference=fit_set)
    start = time.time()
    booster = lgb.train(params, fit_set, num_boost_round=MAX_ROUNDS, valid_sets=[inner_set], valid_names=["inner"],
                        callbacks=[lgb.early_stopping(PATIENCE, verbose=False), lgb.log_evaluation(200)])
    summary = {"model": label, "features": len(columns), "fit_searches": len(fit_sessions), "best_iteration": booster.best_iteration,
               "inner_ndcg_at_10": booster.best_score["inner"]["ndcg@10"], "train_seconds": round(time.time() - start),
               "peak_memory_gb": round(memory_gb()[1], 2)}
    del X, X_inner, fit_set, inner_set
    gc.collect()
    return booster, summary


def score_validation(booster, columns):
    """Long table session_id, label, score for all validation candidates."""
    sid, X, y = load_features("validation", columns)
    frame = pd.DataFrame({"session_id": sid, "label": y,
                          "score": booster.predict(X, num_iteration=booster.best_iteration)})
    del X
    return frame


base_model, base_summary = train_ranker(BASE, label="LambdaMART base G0+G1+G2a")
pd.Series(base_summary).to_frame("E14 base")

,E14 base
model,LambdaMART base G0+G1+G2a
features,10
fit_searches,33810
best_iteration,89
inner_ndcg_at_10,0.3447
train_seconds,16
peak_memory_gb,2.7800


In [6]:
scored = score_validation(base_model, BASE)
scored["pop_lag1d"] = pq.read_table(PROCESSED / f"features_{FEATURES}_validation.parquet",
                                    columns=["pop_history_lag1d"]).column(0).to_numpy()
per_search = {
    "LambdaMART base": evaluate(scored, "score", expected_sizes=validation_sizes),
    "lagged history popularity": evaluate(scored.assign(ref=scored.pop_lag1d.fillna(0).astype(float)), "ref",
                                          expected_sizes=validation_sizes),
    "random order": evaluate(scored.assign(constant=0.0), "constant", expected_sizes=validation_sizes),
}
all_ids = per_search["random order"].index
clean_ids = all_ids[~searches.final_gap_clamped.reindex(all_ids).to_numpy()]

# Sanity check: the reference reproduces its registry value (E13, validation_clean).
registry = pd.read_csv(RESULTS)
logged_ref = registry.query("experiment == 'E13' and model == 'history popularity, lag 1 day' "
                            "and split == 'validation_clean'").ndcg_at_10.iloc[0]
assert abs(per_search["lagged history popularity"].ndcg.loc[clean_ids].mean() - logged_ref) < 1e-4

rows = {}
for population, ids in [("validation_clean", clean_ids), ("validation (all)", all_ids)]:
    for name, per in per_search.items():
        row = summarise(per.loc[ids]).iloc[0].to_dict()
        if name != "lagged history popularity":
            d = paired_difference(per.loc[ids], per_search["lagged history popularity"].loc[ids])
            row.update(vs_lagged_pop=d[0], vs_lagged_pop_low=d[1], vs_lagged_pop_high=d[2])
        rows[(population, name)] = row
results = pd.DataFrame(rows).T
results["searches"] = results.searches.astype(int)
results

searches  ndcg_at_10  ndcg_ci_low  \
validation_clean LambdaMART base               18844      0.3470       0.3421   
                 lagged history popularity     18844      0.3374       0.3327   
                 random order                  18844      0.0842       0.0828   
validation (all) LambdaMART base               19772      0.3459       0.3414   
                 lagged history popularity     19772      0.3369       0.3324   
                 random order                  19772      0.0845       0.0830   

                                            ndcg_ci_high  recall_at_10    mrr  \
validation_clean LambdaMART base                  0.3517        0.5345 0.3447   
                 lagged history popularity        0.3423        0.5212 0.3352   
                 random order                     0.0858        0.1587 0.1026   
validation (all) LambdaMART base                  0.3505        0.5342 0.3420   
                 lagged history popularity        0.3414        0.5212 0.3332   
                 random order                     0.0860        0.1599 0.1020   

                                            vs_lagged_pop  vs_lagged_pop_low  \
validation_clean LambdaMART base                   0.0096             0.0076   
                 lagged history popularity            NaN                NaN   
                 random order                     -0.2533            -0.2575   
validation (all) LambdaMART base                   0.0089             0.0071   
                 lagged history popularity            NaN                NaN   
                 random order                     -0.2525            -0.2570   

                                            vs_lagged_pop_high  
validation_clean LambdaMART base                        0.0114  
                 lagged history popularity                 NaN  
                 random order                          -0.2487  
validation (all) LambdaMART base                        0.0107  
                 lagged history popularity                 NaN  
                 random order                          -0.2481

### By segment (validation_clean)

Popularity is much weaker in long, deep-scroll lists (E12), so list length is the segment to watch; activity in the last 7 days separates shoppers with fresh history from returning ones. Each cell is the paired difference base minus lagged popularity within the segment.

In [7]:
info = searches.loc[clean_ids]
segments = {
    "list length": pd.cut(info.n_candidates, [0, 60, 84, 95, 200], labels=["1-60", "61-84", "85-95", "96-200"]),
    "activity in last 7 days": pd.Series(np.where(info.days_since_last_action < 7, "yes", "no"), index=info.index),
}
rows = []
for segment, labels in segments.items():
    for value in labels.cat.categories if hasattr(labels, "cat") else sorted(labels.unique()):
        ids = labels.index[labels == value]
        d = paired_difference(per_search["LambdaMART base"].loc[ids], per_search["lagged history popularity"].loc[ids])
        rows.append({"segment": segment, "value": value, "searches": len(ids),
                     "lagged_pop": per_search["lagged history popularity"].ndcg.loc[ids].mean(),
                     "base": per_search["LambdaMART base"].ndcg.loc[ids].mean(),
                     "difference": d[0], "low": d[1], "high": d[2]})
pd.DataFrame(rows).set_index(["segment", "value"])

searches  lagged_pop   base  difference  \
segment                 value                                             
list length             1-60        3811      0.4525 0.4716      0.0191   
                        61-84       1693      0.3383 0.3624      0.0241   
                        85-95       8613      0.3666 0.3735      0.0069   
                        96-200      4727      0.1912 0.1927      0.0015   
activity in last 7 days no          2310      0.3334 0.3421      0.0088   
                        yes        16534      0.3380 0.3477      0.0097   

                                   low   high  
segment                 value                  
list length             1-60    0.0145 0.0238  
                        61-84   0.0174 0.0313  
                        85-95   0.0040 0.0097  
                        96-200 -0.0010 0.0042  
activity in last 7 days no      0.0035 0.0141  
                        yes     0.0078 0.0118

Split importance (share of total gain). This shows what the trees use, not how much shoppers value a signal; correlated features share gain arbitrarily.

In [8]:
importance = pd.Series(base_model.feature_importance("gain"), index=BASE)
(importance / importance.sum()).sort_values(ascending=False).rename("gain share").to_frame()

,gain share
pop_history_lag1d,0.8645
bm25,0.0907
title_len,0.0100
query_len,0.0094
title_coverage,0.0084
title_jaccard,0.0058
query_in_brand,0.0047
query_in_cate3,0.0040
title_shared,0.0024
in_catalogue,0.0000


### Log to the results registry

Experiment E14, one row per population. Re-running the notebook does not duplicate rows.

In [9]:
NOTES = (f"LightGBM lambdarank, features {FEATURES} G0+G1+G2a, linear gains, trunc 20, 31 leaves, lr 0.05, "
         f"min leaf 500, inner early stopping (best iteration {base_summary['best_iteration']}), seed {MODEL_SEED}")
logged = pd.read_csv(RESULTS)
for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
    if ((logged.experiment == "E14") & (logged.model == "LambdaMART base G0+G1+G2a") & (logged.split == split)).any():
        continue
    log_result("E14", "LambdaMART base G0+G1+G2a", split, per_search["LambdaMART base"].loc[ids], notes=NOTES)
pd.read_csv(RESULTS).query("experiment == 'E14'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
41,2026-10-27,E14,LambdaMART base G0+G1+G2a,validation_clean,18844,0.3470,0.3421,0.3517,0.5345,0.3447,282a119,"LightGBM lambdarank, features v3 G0+G1+G2a, li..."
42,2026-10-27,E14,LambdaMART base G0+G1+G2a,validation,19772,0.3459,0.3414,0.3505,0.5342,0.3420,282a119,"LightGBM lambdarank, features v3 G0+G1+G2a, li..."


**Reading the result (E14).**
- The trained base reaches **0.347** NDCG@10 on `validation_clean`, **+0.0096 [0.0076, 0.0114]** over lagged history popularity (0.337); all searches give +0.0089. The success criterion (lower bound above +0.003) is met, so grouping, gains and evaluator behave as expected and the base is the reference for personalisation.
- The gain comes from lists of up to 84 candidates (+0.019 to +0.024); in lists of more than 95 candidates the base is not distinguishable from popularity (+0.0015 [-0.0010, 0.0042]). Activity in the last 7 days makes little difference (+0.009 vs +0.010), as expected for a model without personal history.
- Lagged popularity carries 86% of the split gain, BM25 9%: relevance mainly reorders products of similar popularity. Early stopping chose 89 trees; with ten mostly weak features there is little to learn.
- Memory: the 97 features of the 37,567 training searches take 1.3 GB as float32; with the LightGBM dataset and the validation features the process peaked at 2.8 GB. Full-feature ablations on the dev subsample fit comfortably; all training searches (about 11.3M rows, 4.4 GB before LightGBM's dataset) only if the learning curve asks for them.

## 4. E15: full base with same-day popularity (G0 + G1 + G2a + G2b)

G2b adds the same-day popularity signals: training exposures, the out-of-fold engagement rate and history popularity without the one-day lag. All lists are from one day, so other shoppers' same-day behaviour may come after the search being ranked: the full base is an **optimistic, non-deployable reference**, never a candidate model. It shows how much a model would gain from information it could not have at search time.

In [10]:
def compare(name, against=("LambdaMART base", "lagged history popularity")):
    """Summary of `name` on both populations with paired differences against the references."""
    rows = {}
    for population, ids in [("validation_clean", clean_ids), ("validation (all)", all_ids)]:
        row = summarise(per_search[name].loc[ids]).iloc[0].to_dict()
        for other in against:
            d = paired_difference(per_search[name].loc[ids], per_search[other].loc[ids])
            row.update({f"vs {other}": d[0], f"vs {other} low": d[1], f"vs {other} high": d[2]})
        rows[population] = row
    table = pd.DataFrame(rows).T
    table["searches"] = table.searches.astype(int)
    return table


def log_once(experiment, model, split, per, summary):
    """Log one row unless it is already in the registry."""
    logged = pd.read_csv(RESULTS)
    if ((logged.experiment == experiment) & (logged.model == model) & (logged.split == split)).any():
        return
    log_result(experiment, model, split, per,
               notes=(f"LightGBM lambdarank, features {FEATURES}, {summary['features']} inputs, "
                      f"{summary['fit_searches']:,} fit searches, inner early stopping "
                      f"(best iteration {summary['best_iteration']}), configuration as E14, seed {MODEL_SEED}"))


FULL_BASE = BASE + meta["groups"]["G2b_popularity_same_day"]
full_base_model, full_base_summary = train_ranker(FULL_BASE, label="LambdaMART full base G0+G1+G2a+G2b")
per_search["LambdaMART full base"] = evaluate(score_validation(full_base_model, FULL_BASE), "score",
                                              expected_sizes=validation_sizes)
display(pd.Series(full_base_summary).to_frame("E15 full base"))
compare("LambdaMART full base")

[200]	inner's ndcg@10: 0.363027


,E15 full base
model,LambdaMART full base G0+G1+G2a+G2b
features,13
fit_searches,33810
best_iteration,284
inner_ndcg_at_10,0.3646
train_seconds,32
peak_memory_gb,2.7800


,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,vs LambdaMART base,vs LambdaMART base low,vs LambdaMART base high,vs lagged history popularity,vs lagged history popularity low,vs lagged history popularity high
validation_clean,18844,0.3655,0.3602,0.3703,0.5589,0.3618,0.0185,0.0164,0.0206,0.0281,0.0253,0.0307
validation (all),19772,0.3647,0.3601,0.3695,0.5596,0.3591,0.0188,0.0166,0.0211,0.0278,0.0254,0.0303


In [11]:
importance = pd.Series(full_base_model.feature_importance("gain"), index=FULL_BASE)
(importance / importance.sum()).sort_values(ascending=False).rename("gain share").to_frame()

,gain share
pop_history,0.7054
pop_engaged_rate,0.1019
bm25,0.0903
pop_exposures,0.0406
pop_history_lag1d,0.0154
title_len,0.0108
title_coverage,0.0091
title_jaccard,0.0090
query_len,0.0065
query_in_brand,0.0062


In [12]:
for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
    log_once("E15", "LambdaMART full base G0+G1+G2a+G2b (same-day, not deployable)", split,
             per_search["LambdaMART full base"].loc[ids], full_base_summary)
pd.read_csv(RESULTS).query("experiment == 'E15'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
43,2026-10-27,E15,"LambdaMART full base G0+G1+G2a+G2b (same-day, ...",validation_clean,18844,0.3655,0.3602,0.3703,0.5589,0.3618,85de125,"LightGBM lambdarank, features v3, 13 inputs, 3..."
44,2026-10-27,E15,"LambdaMART full base G0+G1+G2a+G2b (same-day, ...",validation,19772,0.3647,0.3601,0.3695,0.5596,0.3591,85de125,"LightGBM lambdarank, features v3, 13 inputs, 3..."


**Reading the result (E15).**
- The full base reaches **0.366** on `validation_clean`, **+0.0185 [0.0164, 0.0206]** over the deployable base: about twice the expected gain. As single signals, lagged and unlagged history popularity differ by only 0.004 (0.337 vs 0.341); combined in a model, the same-day signals are worth much more.
- Unlagged history popularity takes 71% of the split gain and the lagged version almost none: where both are available the trees prefer the version that includes same-day events. Which of the three G2b signals carries the gain is not separated here (correlated features share gain).
- The full base stays a non-deployable, optimistic reference: other shoppers' same-day behaviour can come after the search being ranked. It shows that information from the day of the search is valuable, which is exactly what a deployed ranker cannot use.

## 5. E16: learning curve of the deployable full model

The **deployable full model** uses every v3 group except G2b (94 features): the base plus history (H1-H5), context (C1, C2), query-history (Q1) and source (S1) features. It is trained on nested random subsets of the fit searches (10,000, 20,000 and all 33,810; seed 4222), always with the same inner early-stopping set, and scored on the same validation searches.

Two questions: (1) does the model still gain from more data at the size of the dev subsample (decision rule, fixed before running: if all fit searches minus 20,000 is above +0.003 with a paired interval excluding 0, the features for all training searches are built right after the first ablations); (2) as a first look, how much does personal history and context add over the deployable base? The ablations then take this gain apart group by group. All numbers are screening results; the report quotes the finalists retrained on all training searches.

In [13]:
DEPLOYABLE_FULL = [f for f in ALL_FEATURES if f not in meta["groups"]["G2b_popularity_same_day"]]
curve_order = np.random.default_rng(SEED).permutation(fit_ids)
CURVE_SIZES = [10_000, 20_000, len(fit_ids)]
curve_summaries = {}
for n in CURVE_SIZES:
    booster, summary = train_ranker(DEPLOYABLE_FULL, label=f"deployable full, {n:,} fit searches",
                                    fit_sessions=np.sort(curve_order[:n]))
    name = f"deployable full, {n:,} fit searches"
    per_search[name] = evaluate(score_validation(booster, DEPLOYABLE_FULL), "score", expected_sizes=validation_sizes)
    curve_summaries[n] = summary
    if n == len(fit_ids):
        full_model = booster
FULL_NAME = f"deployable full, {len(fit_ids):,} fit searches"

rows = []
for i, n in enumerate(CURVE_SIZES):
    name = f"deployable full, {n:,} fit searches"
    per = per_search[name].loc[clean_ids]
    row = {"fit_searches": n, "best_iteration": curve_summaries[n]["best_iteration"],
           "train_seconds": curve_summaries[n]["train_seconds"], "ndcg_at_10": per.ndcg.mean()}
    d = paired_difference(per, per_search["LambdaMART base"].loc[clean_ids])
    row.update(vs_base=d[0], vs_base_low=d[1], vs_base_high=d[2])
    if i:
        previous = per_search[f"deployable full, {CURVE_SIZES[i - 1]:,} fit searches"].loc[clean_ids]
        d = paired_difference(per, previous)
        row.update(vs_previous=d[0], vs_previous_low=d[1], vs_previous_high=d[2])
    rows.append(row)
curve = pd.DataFrame(rows).set_index("fit_searches")
print(f"peak memory so far {memory_gb()[1]:.2f} GB")
curve

[200]	inner's ndcg@10: 0.376629


[200]	inner's ndcg@10: 0.379598


[200]	inner's ndcg@10: 0.380939


[400]	inner's ndcg@10: 0.381844


peak memory so far 2.78 GB


,best_iteration,train_seconds,ndcg_at_10,vs_base,vs_base_low,vs_base_high,vs_previous,vs_previous_low,vs_previous_high
fit_searches,,,,,,,,,
10000,231,12,0.3841,0.0371,0.0341,0.0402,NaN,NaN,NaN
20000,169,19,0.3853,0.0383,0.0354,0.0412,0.0012,-0.0002,0.0027
33810,361,124,0.3894,0.0423,0.0395,0.0452,0.0041,0.0028,0.0053


In [14]:
compare(FULL_NAME)

,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,vs LambdaMART base,vs LambdaMART base low,vs LambdaMART base high,vs lagged history popularity,vs lagged history popularity low,vs lagged history popularity high
validation_clean,18844,0.3894,0.3844,0.3946,0.5726,0.3953,0.0423,0.0395,0.0452,0.0519,0.0488,0.0551
validation (all),19772,0.4039,0.3987,0.4091,0.5853,0.4081,0.0580,0.0549,0.0613,0.0670,0.0635,0.0704


### Where the deployable full model gains (validation_clean)

Paired difference full model minus deployable base within each segment. Personal history should matter mostly for shoppers active in the last 7 days (E09: personal single signals +0.107 vs +0.027 over random).

In [15]:
rows = []
for segment, labels in segments.items():
    for value in labels.cat.categories if hasattr(labels, "cat") else sorted(labels.unique()):
        ids = labels.index[labels == value]
        d = paired_difference(per_search[FULL_NAME].loc[ids], per_search["LambdaMART base"].loc[ids])
        rows.append({"segment": segment, "value": value, "searches": len(ids),
                     "base": per_search["LambdaMART base"].ndcg.loc[ids].mean(),
                     "full": per_search[FULL_NAME].ndcg.loc[ids].mean(),
                     "difference": d[0], "low": d[1], "high": d[2]})
pd.DataFrame(rows).set_index(["segment", "value"])

searches   base   full  difference    low  \
segment                 value                                               
list length             1-60        3811 0.4716 0.5061      0.0344 0.0282   
                        61-84       1693 0.3624 0.4019      0.0394 0.0296   
                        85-95       8613 0.3735 0.4212      0.0477 0.0432   
                        96-200      4727 0.1927 0.2327      0.0400 0.0354   
activity in last 7 days no          2310 0.3421 0.3630      0.0209 0.0150   
                        yes        16534 0.3477 0.3930      0.0453 0.0423   

                                 high  
segment                 value          
list length             1-60   0.0408  
                        61-84  0.0484  
                        85-95  0.0526  
                        96-200 0.0448  
activity in last 7 days no     0.0267  
                        yes    0.0484

Split gain by feature group (share of the total). A description of what the trees use, not a ranking of signals by value: correlated groups share gain arbitrarily, and the ablations, not importances, decide which groups matter.

In [16]:
group_of = {f: g for g, features in meta["groups"].items() for f in features}
importance = pd.Series(full_model.feature_importance("gain"), index=DEPLOYABLE_FULL)
by_group = importance.groupby(importance.index.map(group_of)).sum()
display((by_group / by_group.sum()).sort_values(ascending=False).rename("gain share").to_frame())
(importance / importance.sum()).sort_values(ascending=False).head(15).rename("gain share, top 15 features").to_frame()

,gain share
G2a_popularity_deployable,0.6255
H1_pooled,0.1758
G1_relevance,0.1002
H4_match_recency,0.0388
C2_session_context,0.0103
H2_by_type,0.0102
C1_history_volume,0.0093
H5_metadata_coverage,0.0083
S1_history_source,0.0061
Q1_query_history,0.0056


,"gain share, top 15 features"
pop_history_lag1d,0.6255
pooled_product,0.1639
bm25,0.0678
days_since_shop,0.0211
days_since_product,0.0117
title_len,0.0086
title_jaccard,0.0079
pooled_L3,0.0072
days_since_last_action,0.0062
days_since_brand,0.0060


In [17]:
for n in CURVE_SIZES:
    name = f"deployable full, {n:,} fit searches"
    log_once("E16", f"LambdaMART deployable full (no G2b), {n:,} fit searches", "validation_clean",
             per_search[name].loc[clean_ids], curve_summaries[n])
log_once("E16", f"LambdaMART deployable full (no G2b), {len(fit_ids):,} fit searches", "validation",
         per_search[FULL_NAME].loc[all_ids], curve_summaries[len(fit_ids)])
pd.read_csv(RESULTS).query("experiment == 'E16'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
45,2026-10-27,E16,"LambdaMART deployable full (no G2b), 10,000 fi...",validation_clean,18844,0.3841,0.3791,0.3892,0.5674,0.3897,85de125,"LightGBM lambdarank, features v3, 94 inputs, 1..."
46,2026-10-27,E16,"LambdaMART deployable full (no G2b), 20,000 fi...",validation_clean,18844,0.3853,0.3805,0.3907,0.5677,0.3911,85de125,"LightGBM lambdarank, features v3, 94 inputs, 2..."
47,2026-10-27,E16,"LambdaMART deployable full (no G2b), 33,810 fi...",validation_clean,18844,0.3894,0.3844,0.3946,0.5726,0.3953,85de125,"LightGBM lambdarank, features v3, 94 inputs, 3..."
48,2026-10-27,E16,"LambdaMART deployable full (no G2b), 33,810 fi...",validation,19772,0.4039,0.3987,0.4091,0.5853,0.4081,85de125,"LightGBM lambdarank, features v3, 94 inputs, 3..."


**Reading the result (E16).**
- **Personal history and context add +0.042 [0.040, 0.045]** to the deployable base: 0.389 on `validation_clean` (all searches 0.404, +0.058: the clamped searches inflate it, as expected from E13). Within the expectation (+0.02 to +0.05). The deployable full model is also above the non-deployable full base (0.366).
- The gain is about twice as large for shoppers active in the last 7 days (+0.045 vs +0.021) and, unlike the base, it also holds in long lists (more than 95 candidates: +0.040).
- **Learning curve:** 10,000 to 20,000 fit searches +0.0012 [-0.0002, 0.0027], 20,000 to 33,810 **+0.0041 [0.0028, 0.0053]**. The decision rule fixed before running is met, so the features for all training searches are built right after the first ablations (about 30 Oct). The steps are uneven (best iterations 231, 169, 361 with one seed), so the curve shows that more data still helps, not how much.
- Lagged popularity still takes 63% of the split gain, pooled history 18% (almost all exact-product matches), match recency 4%; context, type and window groups get about 1% each. Importances do not say which groups matter: the ablations decide that.